# Differentiable Real-Time Phase Reconstruction for Non-Uniform Filterbanks

## Tutorial: Understanding Diff-RTPGHI Step by Step

This notebook walks through the key ideas behind the Diff-RTPGHI algorithm:

1. **Non-uniform filterbank analysis/synthesis** — how channels with different hop sizes tile the time-frequency plane
2. **Phase gradient estimation** — derivative-filter vs. magnitude-based approaches
3. **Heap-based vs. fixed-order integration** — why replacing the heap makes the algorithm differentiable
4. **Streaming formulation** — causal, frame-by-frame processing
5. **Gradient flow verification** — confirming that meaningful gradients propagate through Diff-RTPGHI

**Paper:** C. Hollomey, *Differentiable Real-Time Phase Reconstruction for Non-Uniform Filterbanks*, IEEE Signal Processing Letters, 2026.

## Setup

Install the LTFAT filterbank library. Update the repository URL once published.

In [ ]:
# ── Setup ──
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                    'cool-frames @ git+https://github.com/allthatsounds/cool-frames.git'],
                   check=True)

import math

import matplotlib.pyplot as plt

import numpy as np

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['figure.dpi'] = 100


In [ ]:
from cool_frames.numpy.filterbanks._core import filterbank, ifilterbank
from cool_frames.numpy.filterbanks._frame import filterbankrealdual
from cool_frames.numpy.filterbanks._utils import normalise_a
from cool_frames.numpy.filters._design import audfilters
from cool_frames.numpy.filters._gabfilters import _comp_tfrfromwin
from cool_frames.numpy.phase._diff_constphase import (
    constphase_nonuniform,
)
from cool_frames.numpy.phase._gla import gla
from cool_frames.numpy.phase._phasegrad import filterbankphasegrad

print('All imports OK.')


## 1. Non-Uniform Filterbank Basics

A non-uniform filterbank has $M$ channels, each with its own hop size $a_m$ and centre frequency $f_{c,m}$. Low-frequency channels have large hop sizes (coarse time resolution, fine frequency resolution) and high-frequency channels have small hop sizes (fine time resolution, coarser frequency resolution). This matches how human hearing works.

Let's build an auditory (ERB-scale) filterbank and visualise its structure.

In [ ]:
fs = 16000  # Sample rate
Ls = 8000   # Signal length (0.5 seconds)
redmul = 8.0  # Redundancy multiplier

# Design the filterbank
g, a, fc_hz, L = audfilters(fs, Ls, redmul=redmul)
M = len(g)
a_norm = normalise_a(a, M)
a_int = np.array([int(a_norm[m, 0]) for m in range(M)])

print(f'Filterbank: {M} channels, signal length L={L}')
print(f'Hop sizes: min={min(a_int)}, max={max(a_int)}')
print(f'Centre frequencies: {fc_hz[0]:.0f} Hz to {fc_hz[-1]:.0f} Hz')

# Number of frames per channel
N = [L // a_int[m] for m in range(M)]
redundancy = sum(N) / L
print(f'Redundancy: {redundancy:.1f}x')

In [ ]:
# Visualise hop sizes and frame counts across channels
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(range(M), a_int, color='steelblue', alpha=0.7)
ax1.set_xlabel('Channel index')
ax1.set_ylabel('Hop size (samples)')
ax1.set_title('Hop size per channel')

ax2.bar(range(M), N, color='coral', alpha=0.7)
ax2.set_xlabel('Channel index')
ax2.set_ylabel('Number of frames')
ax2.set_title('Frames per channel')

plt.tight_layout()
plt.show()

## 2. Analysis and Perfect Reconstruction

With a tight-frame filterbank, we can perfectly reconstruct a signal from its complex coefficients (magnitude + phase). Let's verify this.

In [ ]:
# Create a test signal: sum of sinusoids
t = np.arange(Ls) / fs
sig = 0.5 * np.sin(2*np.pi*440*t) + 0.3 * np.sin(2*np.pi*1000*t) + 0.2 * np.sin(2*np.pi*2500*t)
sig = sig / np.max(np.abs(sig)) * 0.9

# Pad to filterbank length L
sig_padded = np.zeros(L)
sig_padded[:Ls] = sig

# Analysis: get complex coefficients
c = filterbank(sig_padded, g, a_norm, L=L)
print(f'Got {M} channels, shapes: {[np.asarray(ci).shape for ci in c[:3]]}...')

# Synthesis: reconstruct from complex coefficients
gd = filterbankrealdual(g, a_norm, L)
sig_recon = ifilterbank(c, gd, a_norm, Ls=L, real=True)
sig_recon = np.real(sig_recon[:Ls])

# Measure reconstruction error
err = np.max(np.abs(sig - sig_recon))
print(f'Perfect reconstruction error: {err:.2e} (should be ~1e-14)')

## 3. The Phase Retrieval Problem

Now suppose we only have the **magnitudes** — the phases are lost. This happens in many audio processing pipelines (spectral modification, neural network prediction, etc.). How well can we reconstruct the signal from magnitudes alone?

In [ ]:
# Discard phases, keep only magnitudes
s_list = [np.abs(np.asarray(ci).ravel()) for ci in c]

# Reconstruct with zero phase (worst case)
c_zero = [s.astype(complex) for s in s_list]
sig_zero = ifilterbank(c_zero, gd, a_norm, Ls=L, real=True)
sig_zero = np.real(sig_zero[:Ls])

# SDR
def sdr(ref, est):
    L = min(len(ref), len(est))
    r, e = ref[:L], est[:L]
    noise = r - e
    return 10 * np.log10(np.sum(r**2) / (np.sum(noise**2) + 1e-30))

print(f'Zero-phase SDR: {sdr(sig, sig_zero):.1f} dB (terrible!)')

# Plot
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t[:2000], sig[:2000], label='Original', alpha=0.8)
ax.plot(t[:2000], sig_zero[:2000], label='Zero phase', alpha=0.6)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.legend()
ax.set_title('Phase matters! Zero-phase reconstruction is useless.')
plt.tight_layout()
plt.show()

## 4. Phase Gradient Estimation

PGHI estimates the phase by integrating the **phase gradient** — the rate of change of phase in time (instantaneous frequency) and frequency (group delay). There are two approaches:

### 4a. Derivative-filter gradients
Convolve the signal with the frequency-derivative of each filter. This gives the **absolute** instantaneous frequency directly. High accuracy but requires access to the filter derivatives.

### 4b. Magnitude-based gradients  
Estimate the instantaneous frequency from **cross-channel log-magnitude ratios** (Eq. 1 in the paper). This only needs the magnitudes themselves — making it suitable for causal/streaming operation. The result is a **relative** frequency offset that must be added to the channel centre frequency.

Let's compare both approaches.

In [ ]:
# 4a. Derivative-filter gradients
tgrad_l, fgrad_l, s_l, c_with_grad = filterbankphasegrad(sig_padded, g, a_norm, L)

# True phase gradients (from the actual complex coefficients)
true_phases = [np.angle(np.asarray(ci).ravel()) for ci in c]

# Pick a channel with many frames to visualise
ch = M // 2  # Middle channel
true_if = np.diff(np.unwrap(true_phases[ch])) / a_int[ch]  # True inst. freq
est_if = np.asarray(tgrad_l[ch]).ravel() * np.pi  # Derivative-filter estimate

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(true_if[:200], label='True IF', alpha=0.8)
ax.plot(est_if[:200], label='Derivative-filter estimate', alpha=0.6, linestyle='--')
ax.set_xlabel('Frame index')
ax.set_ylabel('Instantaneous frequency (rad/sample)')
ax.set_title(f'Channel {ch}: derivative-filter gradient vs true IF')
ax.legend()
plt.tight_layout()
plt.show()

# Error statistics
n_compare = min(len(true_if), len(est_if))
if_err = np.abs(true_if[:n_compare] - est_if[:n_compare])
print(f'Channel {ch}: mean IF error = {np.mean(if_err):.4f} rad/sample')

In [ ]:
# 4b. Magnitude-based gradients (used in streaming mode)
# These use cross-channel log-magnitude ratios
fc_norm = np.array(fc_hz) / fs * 2.0  # Normalised centre frequencies

# Compute TFR (time-frequency ratio) per channel
tfr = np.zeros(M)
for m in range(M):
    gm = g[m]
    if 'H' in gm:
        H_vals = np.asarray(gm['H'](L))
        h_time = np.fft.ifft(H_vals)
        h_real = np.real(np.fft.fftshift(h_time))
        gamma = _comp_tfrfromwin(h_real)
        tfr[m] = gamma / L if L > 0 else 0.0

sqtfr = np.sqrt(np.abs(tfr))

# Compute magnitude-based tgrad for one frame
slog = np.array([np.log(np.abs(s_list[m][N[m]//2]) + 1e-30) for m in range(M)])
tgrad_mag = _causal_tgrad_tick(slog, fc_norm, sqtfr, M, L)

print('Magnitude-based tgrad computed for one frame.')
print(f'tgrad range: [{tgrad_mag.min():.4f}, {tgrad_mag.max():.4f}]')
print('After adding fc and scaling: these become absolute instantaneous frequencies.')

## 5. Heap-Based vs. Fixed-Order Phase Integration

This is the core contribution of the paper. Both methods integrate the phase gradient to recover phases, but they differ in **traversal order**:

- **Heap (PGHI/RTPGHI):** BFS from the highest-magnitude coefficient. Data-dependent → non-differentiable.
- **Fixed sort (Diff-RTPGHI):** Process all coefficients in descending magnitude order. The sort order is treated as a constant for gradient computation (straight-through estimator).

In **streaming mode** (one tick at a time), both process only 2M candidates, so the difference is minimal.

In [ ]:
# Run streaming Diff-RTPGHI (fixed-order)
c_fixed, phase_fixed, _, _ = constphase_nonuniform(
    s_list, a_int, fc_norm, tfr, tol=1e-6
)

# Reconstruct and measure SC (round-trip)
def sc_roundtrip(c_target, c_recon, g, a_norm, L):
    gd = filterbankrealdual(g, a_norm, L)
    sig_r = ifilterbank(c_recon, gd, a_norm, Ls=L, real=True)
    sig_r = np.real(sig_r)
    c_re = filterbank(sig_r, g, a_norm, L=L)
    r = np.concatenate([np.abs(np.asarray(ci).ravel()) for ci in c_target])
    e = np.concatenate([np.abs(np.asarray(ci).ravel()) for ci in c_re])
    return 20 * np.log10(np.linalg.norm(r - e) / (np.linalg.norm(r) + 1e-30))

sc_diff = sc_roundtrip(c, c_fixed, g, a_norm, L)
print(f'Streaming Diff-RTPGHI SC: {sc_diff:.1f} dB')

# Reconstruct waveform
sig_diff = ifilterbank(c_fixed, gd, a_norm, Ls=L, real=True)
sig_diff = np.real(sig_diff[:Ls])

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t[:2000], sig[:2000], label='Original', alpha=0.8)
ax.plot(t[:2000], sig_diff[:2000], label='Diff-RTPGHI', alpha=0.6, linestyle='--')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.legend()
ax.set_title('Diff-RTPGHI reconstruction')
plt.tight_layout()
plt.show()

## 6. Why is Diff-RTPGHI Differentiable?

The key insight: the sort order $\sigma = \text{argsort}(-|\mathbf{s}|)$ is treated as a **constant** during backpropagation (straight-through estimator). The phase integration equations themselves are differentiable — they're just trapezoidal sums.

This works because small perturbations to the magnitudes rarely change the sort order. When they do, the swapped elements are adjacent in magnitude and produce nearly identical phase estimates.

Let's verify by computing the Jacobian $\partial\phi / \partial s$ via finite differences.

In [ ]:
# Compute Jacobian of one streaming tick via finite differences

# Set up a realistic state (use a middle tick)
events = []
for m in range(M):
    for n in range(N[m]):
        events.append((n * a_int[m], m, n))
events.sort(key=lambda x: (x[0], x[1]))

# Process up to a target tick to get realistic prev_phase etc.
tick_times = sorted(set(ev[0] for ev in events))
target_tick = tick_times[len(tick_times) // 3]

log_bufs = [np.zeros(3) for _ in range(M)]
frame_counts = np.zeros(M, dtype=int)
prev_phase = np.zeros(M)
prev_tgradw = np.zeros(M)
prev_slog = np.full(M, -100.0)
latest_slog = np.full(M, -100.0)
latest_fgrad = np.zeros(M)

i = 0
while i < len(events):
    t_now = events[i][0]
    batch = []
    while i < len(events) and events[i][0] == t_now:
        batch.append(events[i])
        i += 1

    for (t_ev, m_ev, n_ev) in batch:
        mag_val = abs(s_list[m_ev][n_ev])
        slog_val = math.log(mag_val + np.finfo(float).tiny)
        buf = log_bufs[m_ev]
        buf[0] = buf[1]; buf[1] = buf[2]; buf[2] = slog_val
        frame_counts[m_ev] += 1
        tfr_m = sqtfr[m_ev] ** 2
        cnt = int(frame_counts[m_ev])
        if cnt >= 3:
            fd = (3.0 * buf[2] - 4.0 * buf[1] + buf[0]) / 2.0
        elif cnt >= 2:
            fd = buf[2] - buf[1]
        else:
            fd = 0.0
        latest_slog[m_ev] = slog_val
        latest_fgrad[m_ev] = fd * tfr_m / (2.0 * np.pi)

    tgrad_curr = _causal_tgrad_tick(latest_slog, fc_norm, sqtfr, M, L)
    tgradw_curr = (tgrad_curr + fc_norm) * np.pi
    fgradw_curr = -latest_fgrad * np.pi
    time_curr = np.array([t_now] * M, dtype=float)
    time_prev = np.full(M, max(t_now - 1.0, 0.0))

    phase = _fixed_order_phase_tick(
        prev_slog, latest_slog, prev_tgradw, tgradw_curr,
        fgradw_curr, fc_norm, prev_phase, 1e-6, M, time_prev, time_curr
    )
    prev_phase = phase.copy()
    prev_tgradw = tgradw_curr.copy()
    prev_slog = latest_slog.copy()

    if t_now >= target_tick:
        break

# Now compute Jacobian at this tick
eps = 1e-5
phase_ref = _fixed_order_phase_tick(
    prev_slog, latest_slog, prev_tgradw, tgradw_curr,
    fgradw_curr, fc_norm, prev_phase, 1e-6, M, time_prev, time_curr
)

jacobian = np.zeros((M, M))
for j in range(M):
    slog_pert = latest_slog.copy()
    slog_pert[j] += eps
    tgrad_pert = _causal_tgrad_tick(slog_pert, fc_norm, sqtfr, M, L)
    tgradw_pert = (tgrad_pert + fc_norm) * np.pi
    phase_pert = _fixed_order_phase_tick(
        prev_slog, slog_pert, prev_tgradw, tgradw_pert,
        fgradw_curr, fc_norm, prev_phase, 1e-6, M, time_prev, time_curr
    )
    jacobian[:, j] = (phase_pert - phase_ref) / eps

grad_mag = np.sqrt(np.sum(jacobian**2, axis=0))
print(f'Non-zero gradients: {np.sum(grad_mag > 1e-8)}/{M} channels')
print(f'Gradient magnitude range: [{grad_mag.min():.1f}, {grad_mag.max():.1f}]')

In [ ]:
# Visualise the Jacobian and gradient magnitudes
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Jacobian heatmap
im = axes[0].imshow(jacobian, aspect='auto', cmap='RdBu_r',
                     vmin=-np.percentile(np.abs(jacobian), 95),
                     vmax=np.percentile(np.abs(jacobian), 95))
axes[0].set_xlabel('Input channel $j$')
axes[0].set_ylabel('Output channel $i$')
axes[0].set_title('Jacobian $\\partial\\phi_i / \\partial s_j$')
plt.colorbar(im, ax=axes[0])

# Gradient magnitude per input
grad_norm = grad_mag / (grad_mag.max() + 1e-30)
axes[1].plot(range(M), grad_norm, 'o-', color='#d62728', markersize=4)
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5, label='Heap RTPGHI (undefined)')
axes[1].set_xlabel('Channel index')
axes[1].set_ylabel('$||\\partial\\phi / \\partial s_m||$ (normalised)')
axes[1].set_title('Gradient flow: Diff-RTPGHI vs Heap')
axes[1].legend(fontsize=9)

# Magnitudes at this tick
axes[2].bar(range(M), np.exp(latest_slog), color='steelblue', alpha=0.7)
axes[2].set_xlabel('Channel index')
axes[2].set_ylabel('Magnitude')
axes[2].set_title('Input magnitudes at this tick')

plt.tight_layout()
plt.show()

print('\nKey observation: Diff-RTPGHI has non-zero gradients for ALL channels.')
print('The heap-based RTPGHI would have zero/undefined gradients because')
print('argsort has zero gradient almost everywhere.')

## 7. Improving Quality with fGLA

Diff-RTPGHI provides a single-pass phase estimate. We can use it as initialisation for the iterative fast Griffin-Lim algorithm (fGLA) to get even better quality.

In [ ]:
# Batch PGHI (heap-based, derivative-filter gradients) for comparison
tgrad_l, fgrad_l, s_l, c_pghi_raw = filterbankphasegrad(sig_padded, g, a_norm, L)

# Use PGHI output as fGLA initialisation
c_fgla, _, _, _ = gla(
    c_pghi_raw, g, a_norm, L=L, real=True,
    maxit=100, method='fgla', startphase='input'
)

sc_fgla = sc_roundtrip(c, c_fgla, g, a_norm, L)
print(f'fGLA (PGHI init, 100 it.) SC: {sc_fgla:.1f} dB')
print(f'Streaming Diff-RTPGHI SC:     {sc_diff:.1f} dB')
print('\nfGLA is much better but requires 100 iterations and is not real-time.')

## Summary

| Property | Heap RTPGHI | **Diff-RTPGHI** | fGLA |
|----------|-------------|-----------------|------|
| Single-pass | ✓ | ✓ | ✗ (iterative) |
| Causal/streaming | ✓ | ✓ | ✗ |
| Differentiable | ✗ | **✓** | ✗ |
| Quality gap | reference | <0.2 dB | better |

**Diff-RTPGHI trades negligible quality loss (<0.2 dB in streaming mode) for full differentiability**, enabling integration into end-to-end trainable pipelines.

See **Notebook 2** for full experiment reproduction and **Notebook 3** for an end-to-end training demo.